# Task 3: rating prediction with time (MSE ↓, 0-ratings skipped)

**Shipped:** bias model α + β_u + β_i (ALS), reg_u 3, reg_i 5, 25 epochs, on nonzero rows.
**Val 0.7449 | public 0.7626 | tier 2.0** (2.5 ≤ 0.75, 3.0 ≤ 0.73, 3.5 ≤ 0.71).
Public runs ~0.018 *worse* than validation (the only task where it does): check how the test
was drawn before trusting validation gains.

**Earlier results (old notebook):** time-bias β_i + β_{i,bin(t)} (reg_u 3, reg_i 5, 10 bins,
reg_ib 50) → 0.7406; time one-hot ridge on residuals → 0.7437; same-(user, day) residual → 0.7392.

**This notebook:** port Task 1's `StackedRating` (bias + item-kNN + stats, −6% on Task 1), try
`TimeBiasModel` as the base, and add a same-session `FeatureSource`.

In [2]:
%load_ext autoreload
%autoreload 2

from assignment_1.modules.util import calc_mse, drop_zero_ratings, load_csv, random_split
from assignment_1.modules.models import (
    BiasModel,
    FeatureSource,
    ItemKNN,
    RatingModel,
    RatingStats,
    StackedRating,
    TimeBiasModel,
)
from sklearn.ensemble import HistGradientBoostingRegressor
import pandas as pd
import numpy as np
from typing import Self



DATA_DIR = "../data"  # notebook runs from notebooks/

train = load_csv("tasks_3_4_train.csv", DATA_DIR)
test = load_csv("task_3_test.csv", DATA_DIR)
nonzero = drop_zero_ratings(train)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data checks
Timestamps outside 2005–2018 are clipped by the models. The second cell asks how the test was
drawn: if test rows are mostly *after* each user's training history, a random validation split
is optimistic (which would explain public being worse than validation).

In [3]:
for name, df in [("train", train), ("test", test)]:
    t = pd.to_datetime(df.timestamp, unit="s")
    odd = (t < "2005-01-01") | (t > "2018-12-31")
    print(f"{name}: {odd.sum():,} rows outside 2005–2018 ({odd.mean():.3%})")
print(f"test columns: {test.columns.tolist()}")

train: 1,944 rows outside 2005–2018 (0.136%)
test: 30 rows outside 2005–2018 (0.150%)
test columns: ['user', 'item', 'timestamp', 'rating']


In [4]:
t_last = train.groupby("user")["timestamp"].max()
t_first = train.groupby("user")["timestamp"].min()
after = (test["timestamp"] > test["user"].map(t_last)).mean()
inside = ((test["timestamp"] >= test["user"].map(t_first)) & (
    test["timestamp"] <= test["user"].map(t_last))).mean()
print(
    f"test rows after the user's last train row: {after:.1%} | inside the user's train span: {inside:.1%}")
q = [0.1, 0.5, 0.9]
print("timestamp quantiles  train:", pd.to_datetime(
    train.timestamp.quantile(q), unit="s").dt.date.tolist())
print("                     test: ", pd.to_datetime(
    test.timestamp.quantile(q), unit="s").dt.date.tolist())

test rows after the user's last train row: 2.6% | inside the user's train span: 94.7%
timestamp quantiles  train: [datetime.date(2011, 7, 25), datetime.date(2014, 9, 27), datetime.date(2017, 2, 14)]
                     test:  [datetime.date(2011, 7, 17), datetime.date(2014, 9, 18), datetime.date(2017, 2, 11)]


## Same-session feature source
Residual mean of the user's training rows on the same day / within ±7 days of the row's time.
Data check (Task 1): P(same rating) 0.55 within an hour vs 0.43 after a month.
Lives here until it proves useful; then move it to `models/rating/`.

In [ ]:
class SessionResidual(FeatureSource):
    """Shrunk mean residual (r − b̂) of the user's training rows near the row's timestamp."""

    def __init__(self, windows=(("day", 86400), ("week", 7 * 86400)), shrink: float = 1.0) -> None:
        self.windows, self.shrink = windows, shrink

    def fit(self, train: pd.DataFrame, base: RatingModel) -> Self:
        res = train["rating"].to_numpy() - base.predict(train)
        self.tables = {}
        for name, w in self.windows:
            key = pd.MultiIndex.from_arrays(
                [train["user"], train["timestamp"] // w])
            self.tables[name] = (w, pd.Series(res, index=key).groupby(
                level=[0, 1]).agg(["sum", "size"]))
        return self

    def features(self, rows: pd.DataFrame) -> pd.DataFrame:
        out = {}
        for name, (w, tbl) in self.tables.items():
            s = tbl.reindex(pd.MultiIndex.from_arrays(
                [rows["user"], rows["timestamp"] // w]))
            out[f"sess_{name}"] = (
                s["sum"] / (s["size"] + self.shrink)).fillna(0).to_numpy()
            out[f"log_n_{name}"] = np.log1p(s["size"].fillna(0).to_numpy())
        return pd.DataFrame(out)

## Validation split + yardsticks
Same split as before; `BiasModel` must reproduce 0.7449 and `TimeBiasModel` 0.7406.

In [6]:
tr, val = random_split(train, val_frac=0.25, seed=0)
tr, val = drop_zero_ratings(tr), drop_zero_ratings(val)
y_val = val["rating"].to_numpy()

BIAS = dict(reg_u=3.0, reg_i=5.0, n_epochs=25)
TIME = dict(reg_u=3.0, reg_i=5.0, reg_ib=50.0, n_bins=10)

bias_m = BiasModel(**BIAS).fit(tr)
time_m = TimeBiasModel(**TIME).fit(tr)
pd.Series({
    "bias (expect 0.7449)": calc_mse(y_val, bias_m.predict(val)),
    "time bias (expect 0.7406)": calc_mse(y_val, time_m.predict(val)),
}).round(4)

bias (expect 0.7449)         0.7449
time bias (expect 0.7406)    0.7406
dtype: float64

## E1. Stacks on an outer holdout
Each `StackedRating` trains its regressor on its own internal split, so all are scored on an
outer 10% holdout none of them see. Compare: base model (bias vs time bias) and sources
(kNN + stats, then + session).

In [7]:
outer_tr, outer_val = random_split(nonzero, val_frac=0.1, seed=1)
y_out = outer_val["rating"].to_numpy()
GB = HistGradientBoostingRegressor(random_state=0)

configs = {
    "bias only": None,
    "stack: bias + kNN + stats": (BiasModel(**BIAS), [ItemKNN(), RatingStats()]),
    "stack: time + kNN + stats": (TimeBiasModel(**TIME), [ItemKNN(), RatingStats()]),
    "stack: time + kNN + stats + session": (TimeBiasModel(**TIME), [ItemKNN(), RatingStats(), SessionResidual()]),
}
results = {}
for name, cfg in configs.items():
    if cfg is None:
        pred = BiasModel(**BIAS).fit(outer_tr).predict(outer_val)
    else:
        base, sources = cfg
        pred = StackedRating(base=base, sources=sources, regressor=GB).fit(
            outer_tr).predict(outer_val)
    results[name] = calc_mse(y_out, pred)
    print(f"{name:<40} {results[name]:.4f}")

res = pd.Series(results)
(res / res["bias only"] - 1).mul(100).round(2).rename("vs bias %")

bias only                                0.7369
stack: bias + kNN + stats                0.7066
stack: time + kNN + stats                0.7024
stack: time + kNN + stats + session      0.7019


bias only                              0.00
stack: bias + kNN + stats             -4.12
stack: time + kNN + stats             -4.68
stack: time + kNN + stats + session   -4.75
Name: vs bias %, dtype: float64

## Next
- If the test is mostly *after* users' histories (data check), rebuild validation as a per-user
  time holdout, so gains are measured the way the test is drawn.
- Ship the best stack in `run_task3` (same shape as `run_task1`), `make check`, upload.
- Targets: 3.0 ≤ 0.73 public (~ −4.3% from 0.7626); 3.5 ≤ 0.71 (~ −6.9%).